# E-Commerce Sales & Customer Analytics

## 02 — Data Cleaning & Preparation

This notebook transforms the raw Online Retail II transaction data
into a reliable analytical dataset for sales, product, customer,
geographic, and time-based analysis.

### Cleaning Objectives

- Load and combine both years of transaction data
- Standardize column names and data types
- Identify and handle cancelled transactions
- Remove exact duplicate records
- Validate quantities and prices
- Handle missing customer information appropriately
- Create business metrics
- Create analytical date dimensions
- Validate the cleaned dataset
- Export the analytical dataset

In [1]:
import pandas as pd
import numpy as np

print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)

Pandas: 3.0.5
NumPy: 2.4.6


In [2]:
file_path = "../data/online_retail_II.xlsx"

df_2009 = pd.read_excel(
    file_path,
    sheet_name="Year 2009-2010"
)

df_2010 = pd.read_excel(
    file_path,
    sheet_name="Year 2010-2011"
)

print("2009–2010:", df_2009.shape)
print("2010–2011:", df_2010.shape)

2009–2010: (525461, 8)
2010–2011: (541910, 8)


In [3]:
df_raw = pd.concat(
    [df_2009, df_2010],
    ignore_index=True
)

print("Raw combined shape:", df_raw.shape)

Raw combined shape: (1067371, 8)


In [4]:
df_raw.columns = (
    df_raw.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

df_raw.columns

Index(['invoice', 'stockcode', 'description', 'quantity', 'invoicedate',
       'price', 'customer_id', 'country'],
      dtype='str')

In [5]:
cleaning_audit = {
    "raw_rows": len(df_raw),
    "raw_columns": df_raw.shape[1],
    "duplicate_rows": df_raw.duplicated().sum(),
    "missing_customer_id": df_raw["customer_id"].isna().sum(),
    "cancelled_invoices": df_raw["invoice"].astype(str).str.startswith("C").sum(),
    "negative_quantity": (df_raw["quantity"] < 0).sum(),
    "zero_quantity": (df_raw["quantity"] == 0).sum(),
    "zero_price": (df_raw["price"] == 0).sum()
}

cleaning_audit

{'raw_rows': 1067371,
 'raw_columns': 8,
 'duplicate_rows': np.int64(34335),
 'missing_customer_id': np.int64(243007),
 'cancelled_invoices': np.int64(19494),
 'negative_quantity': np.int64(22950),
 'zero_quantity': np.int64(0),
 'zero_price': np.int64(6202)}

In [6]:
df_raw["invoice"] = df_raw["invoice"].astype(str).str.strip()
df_raw["stockcode"] = df_raw["stockcode"].astype(str).str.strip()
df_raw["description"] = df_raw["description"].astype("string").str.strip()
df_raw["country"] = df_raw["country"].astype("string").str.strip()

df_raw["invoicedate"] = pd.to_datetime(
    df_raw["invoicedate"],
    errors="coerce"
)

df_raw["quantity"] = pd.to_numeric(
    df_raw["quantity"],
    errors="coerce"
)

df_raw["price"] = pd.to_numeric(
    df_raw["price"],
    errors="coerce"
)

df_raw["customer_id"] = pd.to_numeric(
    df_raw["customer_id"],
    errors="coerce"
)

df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   invoice      1067371 non-null  str           
 1   stockcode    1067371 non-null  str           
 2   description  1062989 non-null  string        
 3   quantity     1067371 non-null  int64         
 4   invoicedate  1067371 non-null  datetime64[us]
 5   price        1067371 non-null  float64       
 6   customer_id  824364 non-null   float64       
 7   country      1067371 non-null  string        
dtypes: datetime64[us](1), float64(2), int64(1), str(2), string(2)
memory usage: 65.1 MB


In [7]:
before_duplicates = len(df_raw)

df_clean = df_raw.drop_duplicates().copy()

after_duplicates = len(df_clean)

print("Rows before removing duplicates:", f"{before_duplicates:,}")
print("Rows after removing duplicates:", f"{after_duplicates:,}")
print(
    "Duplicates removed:",
    f"{before_duplicates - after_duplicates:,}"
)

Rows before removing duplicates: 1,067,371
Rows after removing duplicates: 1,033,036
Duplicates removed: 34,335


In [8]:
df_clean["is_cancelled"] = (
    df_clean["invoice"]
    .str.upper()
    .str.startswith("C")
)

print(
    "Cancelled transactions:",
    f"{df_clean['is_cancelled'].sum():,}"
)

print(
    "Normal transactions:",
    f"{(~df_clean['is_cancelled']).sum():,}"
)

Cancelled transactions: 19,104
Normal transactions: 1,013,932


In [9]:
df_clean["revenue"] = (
    df_clean["quantity"] *
    df_clean["price"]
)

In [10]:
print(
    "Total transaction value:",
    f"£{df_clean['revenue'].sum():,.2f}"
)

Total transaction value: £18,855,533.70


In [11]:
df_clean["transaction_type"] = np.where(
    df_clean["is_cancelled"],
    "Cancellation",
    "Sale"
)

df_clean["transaction_type"].value_counts()

transaction_type
Sale            1013932
Cancellation      19104
Name: count, dtype: int64

In [12]:
df_clean["year"] = df_clean["invoicedate"].dt.year

df_clean["month"] = df_clean["invoicedate"].dt.month

df_clean["month_name"] = (
    df_clean["invoicedate"]
    .dt.month_name()
)

df_clean["quarter"] = (
    "Q" +
    df_clean["invoicedate"].dt.quarter.astype(str)
)

df_clean["day_of_week"] = (
    df_clean["invoicedate"]
    .dt.day_name()
)

df_clean["date"] = (
    df_clean["invoicedate"]
    .dt.date
)

In [13]:
print("Rows:", f"{len(df_clean):,}")
print("Columns:", df_clean.shape[1])

print("\nMissing values:")
print(df_clean.isna().sum())

print("\nNegative quantities:")
print((df_clean["quantity"] < 0).sum())

print("\nZero prices:")
print((df_clean["price"] == 0).sum())

Rows: 1,033,036
Columns: 17

Missing values:
invoice                  0
stockcode                0
description           4275
quantity                 0
invoicedate              0
price                    0
customer_id         235151
country                  0
is_cancelled             0
revenue                  0
transaction_type         0
year                     0
month                    0
month_name               0
quarter                  0
day_of_week              0
date                     0
dtype: int64

Negative quantities:
22496

Zero prices:
6014


In [14]:
negative_qty = df_clean[df_clean["quantity"] < 0].copy()

print("Negative quantity rows:", f"{len(negative_qty):,}")
print("Cancelled rows among negative quantities:",
      f"{negative_qty['is_cancelled'].sum():,}")

negative_qty[[
    "invoice",
    "stockcode",
    "description",
    "quantity",
    "price",
    "customer_id",
    "is_cancelled",
    "transaction_type"
]].head(20)

Negative quantity rows: 22,496
Cancelled rows among negative quantities: 19,103


,invoice,stockcode,description,quantity,price,customer_id,is_cancelled,transaction_type
178,C489449,22087,PAPER BUNTING WHITE LACE,-12,2.95,16321.0,True,Cancellation
179,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6,1.65,16321.0,True,Cancellation
180,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4,4.25,16321.0,True,Cancellation
181,C489449,21896,POTTING SHED TWINE,-6,2.10,16321.0,True,Cancellation
182,C489449,22083,PAPER CHAIN KIT RETRO SPOT,-12,2.95,16321.0,True,Cancellation
183,C489449,21871,SAVE THE PLANET MUG,-12,1.25,16321.0,True,Cancellation
184,C489449,84946,ANTIQUE SILVER TEA GLASS ETCHED,-12,1.25,16321.0,True,Cancellation
185,C489449,84970S,HANGING HEART ZINC T-LIGHT HOLDER,-24,0.85,16321.0,True,Cancellation
186,C489449,22090,PAPER BUNTING RETRO SPOTS,-12,2.95,16321.0,True,Cancellation
196,C489459,90200A,PURPLE SWEETHEART BRACELET,-3,4.25,17592.0,True,Cancellation


In [15]:
zero_price = df_clean[df_clean["price"] == 0].copy()

print("Zero-price rows:", f"{len(zero_price):,}")
print(
    "Zero-price cancelled rows:",
    f"{zero_price['is_cancelled'].sum():,}"
)

zero_price[[
    "invoice",
    "stockcode",
    "description",
    "quantity",
    "price",
    "customer_id",
    "country",
    "transaction_type"
]].head(20)

Zero-price rows: 6,014
Zero-price cancelled rows: 0


,invoice,stockcode,description,quantity,price,customer_id,country,transaction_type
263,489464,21733,85123a mixed,-96,0.0,NaN,United Kingdom,Sale
283,489463,71477,short,-240,0.0,NaN,United Kingdom,Sale
284,489467,85123A,21733 mixed,-192,0.0,NaN,United Kingdom,Sale
470,489521,21646,<NA>,-50,0.0,NaN,United Kingdom,Sale
3114,489655,20683,<NA>,-44,0.0,NaN,United Kingdom,Sale
3161,489659,21350,<NA>,230,0.0,NaN,United Kingdom,Sale
3162,489660,35956,lost,-1043,0.0,NaN,United Kingdom,Sale
3168,489663,35605A,damages,-117,0.0,NaN,United Kingdom,Sale
3731,489781,84292,<NA>,17,0.0,NaN,United Kingdom,Sale
4296,489806,18010,<NA>,-770,0.0,NaN,United Kingdom,Sale


In [16]:
transaction_summary = (
    df_clean
    .groupby("transaction_type")
    .agg(
        transactions=("invoice", "nunique"),
        rows=("invoice", "size"),
        quantity=("quantity", "sum"),
        revenue=("revenue", "sum")
    )
    .reset_index()
)

transaction_summary

,transaction_type,transactions,rows,quantity,revenue
0,Cancellation,8292,19104,-476819,-1.462051e+06
1,Sale,45336,1013932,10886598,2.031758e+07


In [17]:
cancelled_summary = (
    df_clean[df_clean["is_cancelled"]]
    .groupby("invoice")
    .agg(
        items=("stockcode", "count"),
        quantity=("quantity", "sum"),
        revenue=("revenue", "sum")
    )
    .reset_index()
)

cancelled_summary.head(10)

,invoice,items,quantity,revenue
0,C489449,9,-100,-196.10
1,C489459,13,-38,-148.30
2,C489476,1,-6,-12.60
3,C489503,2,-3,-13.45
4,C489504,1,-6,-15.30
5,C489518,3,-4,-38.40
6,C489524,1,-1,-12.75
7,C489527,3,-21,-43.95
8,C489528,1,-1,-7.95
9,C489530,1,-1,-4.25


In [18]:
cancelled_summary["quantity"].describe()

count     8292.000000
mean       -57.503497
std       1554.531735
min     -87167.000000
25%        -10.000000
50%         -3.000000
75%         -1.000000
max          1.000000
Name: quantity, dtype: float64

In [19]:
df_sales = df_clean[
    (~df_clean["is_cancelled"]) &
    (df_clean["quantity"] > 0) &
    (df_clean["price"] > 0)
].copy()

In [20]:
print("Sales rows:", f"{len(df_sales):,}")
print("Sales invoices:", f"{df_sales['invoice'].nunique():,}")
print("Sales customers:", f"{df_sales['customer_id'].nunique():,}")
print("Sales products:", f"{df_sales['stockcode'].nunique():,}")
print("Sales revenue:", f"£{df_sales['revenue'].sum():,.2f}")

Sales rows: 1,007,913
Sales invoices: 40,077
Sales customers: 5,878
Sales products: 4,916
Sales revenue: £20,476,260.45


In [21]:
sales_quality = pd.DataFrame({
    "Metric": [
        "Rows",
        "Unique invoices",
        "Unique customers",
        "Unique products",
        "Missing descriptions",
        "Missing customer IDs",
        "Negative quantities",
        "Zero prices",
        "Total revenue"
    ],
    "Value": [
        len(df_sales),
        df_sales["invoice"].nunique(),
        df_sales["customer_id"].nunique(),
        df_sales["stockcode"].nunique(),
        df_sales["description"].isna().sum(),
        df_sales["customer_id"].isna().sum(),
        (df_sales["quantity"] < 0).sum(),
        (df_sales["price"] == 0).sum(),
        f"£{df_sales['revenue'].sum():,.2f}"
    ]
})

sales_quality

,Metric,Value
0,Rows,1007913
1,Unique invoices,40077
2,Unique customers,5878
3,Unique products,4916
4,Missing descriptions,0
5,Missing customer IDs,228488
6,Negative quantities,0
7,Zero prices,0
8,Total revenue,"£20,476,260.45"


In [22]:
df_customer = df_sales[
    df_sales["customer_id"].notna()
].copy()

print("Customer transactions:", f"{len(df_customer):,}")
print("Unique customers:", f"{df_customer['customer_id'].nunique():,}")

Customer transactions: 779,425
Unique customers: 5,878


In [23]:
analysis_date = (
    df_customer["invoicedate"].max()
    + pd.Timedelta(days=1)
)

print("Last transaction:", df_customer["invoicedate"].max())
print("RFM analysis date:", analysis_date)

Last transaction: 2011-12-09 12:50:00
RFM analysis date: 2011-12-10 12:50:00


In [24]:
rfm = (
    df_customer
    .groupby("customer_id")
    .agg(
        recency=(
            "invoicedate",
            lambda x: (analysis_date - x.max()).days
        ),
        frequency=("invoice", "nunique"),
        monetary=("revenue", "sum")
    )
    .reset_index()
)

rfm.head()

,customer_id,recency,frequency,monetary
0,12346.0,326,12,77556.46
1,12347.0,2,8,4921.53
2,12348.0,75,5,2019.40
3,12349.0,19,4,4428.69
4,12350.0,310,1,334.40


In [25]:
rfm[["recency", "frequency", "monetary"]].describe()

,recency,frequency,monetary
count,5878.000000,5878.000000,5878.000000
mean,201.331916,6.289384,2955.904095
std,209.338707,13.009406,14440.852688
min,1.000000,1.000000,2.950000
25%,26.000000,1.000000,342.280000
50%,96.000000,3.000000,867.740000
75%,380.000000,7.000000,2248.305000
max,739.000000,398.000000,580987.040000


In [26]:
rfm["r_score"] = pd.qcut(
    rfm["recency"],
    4,
    labels=[4, 3, 2, 1],
    duplicates="drop"
)

rfm["f_score"] = pd.qcut(
    rfm["frequency"].rank(method="first"),
    4,
    labels=[1, 2, 3, 4]
)

rfm["m_score"] = pd.qcut(
    rfm["monetary"].rank(method="first"),
    4,
    labels=[1, 2, 3, 4]
)

In [27]:
rfm["rfm_score"] = (
    rfm["r_score"].astype(int).astype(str)
    + rfm["f_score"].astype(int).astype(str)
    + rfm["m_score"].astype(int).astype(str)
)

rfm.head()

,customer_id,recency,frequency,monetary,r_score,f_score,m_score,rfm_score
0,12346.0,326,12,77556.46,2,4,4,244
1,12347.0,2,8,4921.53,4,4,4,444
2,12348.0,75,5,2019.40,3,3,3,333
3,12349.0,19,4,4428.69,4,3,4,434
4,12350.0,310,1,334.40,2,1,1,211


In [28]:
def segment_customer(row):
    r = int(row["r_score"])
    f = int(row["f_score"])
    m = int(row["m_score"])

    if r >= 3 and f >= 3 and m >= 3:
        return "Champions"

    elif r >= 3 and f >= 2:
        return "Loyal Customers"

    elif r >= 3 and f <= 2:
        return "New / Promising"

    elif r <= 2 and f >= 3:
        return "At Risk"

    elif r <= 2 and f <= 2 and m >= 3:
        return "High Value At Risk"

    else:
        return "Needs Attention"


rfm["segment"] = rfm.apply(segment_customer, axis=1)

rfm["segment"].value_counts()

segment
Champions             1821
Needs Attention       1776
At Risk                880
Loyal Customers        796
New / Promising        336
High Value At Risk     269
Name: count, dtype: int64

In [29]:
segment_summary = (
    rfm
    .groupby("segment")
    .agg(
        customers=("customer_id", "nunique"),
        avg_recency=("recency", "mean"),
        avg_frequency=("frequency", "mean"),
        total_revenue=("monetary", "sum"),
        avg_customer_value=("monetary", "mean")
    )
    .reset_index()
    .sort_values("total_revenue", ascending=False)
)

segment_summary

,segment,customers,avg_recency,avg_frequency,total_revenue,avg_customer_value
1,Champions,1821,29.303130,14.072488,1.318633e+07,7241.258115
0,At Risk,880,277.125000,6.526136,2.165028e+06,2460.258573
3,Loyal Customers,796,36.300251,2.797739,7.760723e+05,974.965254
4,Needs Attention,1776,418.957770,1.393581,5.653163e+05,318.308706
2,High Value At Risk,269,362.710037,2.089219,5.514783e+05,2050.105364
5,New / Promising,336,46.622024,1.000000,1.305788e+05,388.627232


In [30]:
top_customers = (
    rfm
    .sort_values("monetary", ascending=False)
    .head(20)
)

top_customers[
    [
        "customer_id",
        "recency",
        "frequency",
        "monetary",
        "rfm_score",
        "segment"
    ]
]

,customer_id,recency,frequency,monetary,rfm_score,segment
5692,18102.0,1,145,580987.04,444,Champions
2277,14646.0,2,151,528602.52,444,Champions
1789,14156.0,10,156,313437.62,444,Champions
2538,14911.0,1,398,291420.81,444,Champions
5050,17450.0,8,51,244784.25,444,Champions
1331,13694.0,4,143,195640.69,444,Champions
5109,17511.0,3,60,172132.87,444,Champions
4061,16446.0,1,2,168472.50,424,Loyal Customers
4295,16684.0,4,55,147142.77,444,Champions
68,12415.0,24,28,144458.37,444,Champions


In [31]:
customer_kpis = {
    "Total Customers": rfm["customer_id"].nunique(),
    "Total Customer Revenue": rfm["monetary"].sum(),
    "Average Customer Revenue": rfm["monetary"].mean(),
    "Average Purchase Frequency": rfm["frequency"].mean(),
    "Median Recency": rfm["recency"].median(),
    "Champions": (rfm["segment"] == "Champions").sum(),
    "At Risk": (rfm["segment"] == "At Risk").sum()
}

customer_kpis

{'Total Customers': 5878,
 'Total Customer Revenue': np.float64(17374804.268),
 'Average Customer Revenue': np.float64(2955.9040945899965),
 'Average Purchase Frequency': np.float64(6.289384144266758),
 'Median Recency': np.float64(96.0),
 'Champions': np.int64(1821),
 'At Risk': np.int64(880)}